# DA-DAKA verified Colab training

이 notebook은 이미 전처리·독립 audit가 끝난 release만 사용합니다. 재-dedup, 재-split, polygon repair, ROI 재생성은 수행하지 않습니다. Drive upload가 완료되기 전에는 실행하지 마세요. 각 학습 entrypoint도 동일한 release 검증을 다시 수행하므로 검증 셀을 건너뛰어도 잘못된 release로 학습할 수 없습니다.


In [ ]:
# 사용자 입력: Drive upload가 완료된 뒤 두 Drive 경로만 지정하세요.
REPO_URL = 'https://github.com/KiHyeonLee1121/da-daka_Ai.git'
REPO_REF = 'main'
DRIVE_DATASET_ROOT = ''  # 예: /content/drive/MyDrive/da-daka/releases/da-daka-0fe4fc5f136e2a79
DRIVE_RESULTS_ROOT = ''  # 예: /content/drive/MyDrive/da-daka/training-runs
RUN_LABEL = 'baseline-v1'
TASKS = ('panel', 'dirt')
PANEL_RESUME = ''  # 선택: Drive의 .../panel/checkpoints/last.pt
DIRT_RESUME = ''   # 선택: Drive의 .../dirt/checkpoints/last.pt
EXPECTED_VERSION = 'da-daka-0fe4fc5f136e2a79'
EXPECTED_FINGERPRINT = '0fe4fc5f136e2a79240c3ddf7ba731d45a187b044c9a7a9fdf5bff956145a9fe'
LOCAL_REPO = '/content/da-daka_Ai'
LOCAL_DATASET_ROOT = '/content/da_daka_dataset'
LOCAL_RUNS_ROOT = '/content/da_daka_runs'


## 1. GPU fail-fast (Drive mount보다 먼저)


In [ ]:
import json, platform, subprocess, sys
import torch
if not torch.cuda.is_available():
    raise RuntimeError('COLAB GPU REQUIRED: Runtime > Change runtime type에서 GPU를 선택하세요.')
device = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device)
torch.empty((1,), device='cuda')
torch.cuda.synchronize()
print(json.dumps({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': props.name,
    'gpu_memory_gib': round(props.total_memory / 2**30, 2),
    'compute_capability': torch.cuda.get_device_capability(device),
}, indent=2))
subprocess.run(['nvidia-smi'], check=True)


## 2. 최신 repository와 Colab dependencies


In [ ]:
from pathlib import Path
repo = Path(LOCAL_REPO)
if repo.exists():
    if not (repo / '.git').is_dir():
        raise RuntimeError(f'LOCAL_REPO exists but is not a Git checkout: {repo}')
    dirty = subprocess.run(['git', '-C', str(repo), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout
    if dirty.strip():
        raise RuntimeError('Existing Colab checkout is dirty; choose a fresh runtime or inspect it manually.')
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(repo), 'merge', '--ff-only', f'origin/{REPO_REF}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'training/requirements-colab.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo / 'laptop_ai'), '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo / 'training'), '--no-deps'], check=True)
subprocess.run(['da-daka-gpu-doctor'], check=True)
REPO_COMMIT = subprocess.run(['git', '-C', str(repo), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
print('Repository commit:', REPO_COMMIT)


## 3. Google Drive mount와 경로 확인


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
if not DRIVE_DATASET_ROOT.strip() or not DRIVE_RESULTS_ROOT.strip():
    raise RuntimeError('DRIVE_DATASET_ROOT와 DRIVE_RESULTS_ROOT를 첫 셀에 지정하세요.')
drive_dataset = Path(DRIVE_DATASET_ROOT).expanduser().resolve()
drive_results = Path(DRIVE_RESULTS_ROOT).expanduser().resolve()
if not drive_dataset.is_dir():
    raise RuntimeError(f'Drive dataset release root not found: {drive_dataset}')
drive_results.mkdir(parents=True, exist_ok=True)
preflight_dir = drive_results / RUN_LABEL / 'preflight'
preflight_dir.mkdir(parents=True, exist_ok=True)
print('Drive release:', drive_dataset)
print('Drive results:', drive_results / RUN_LABEL)


## 4. Drive release preflight (metadata, counts, canonical fingerprint)

여기서 하나라도 누락되거나 identity가 다르면 `DATASET INCOMPLETE OR WRONG RELEASE`로 중단됩니다.


In [ ]:
subprocess.run([
    'da-daka-verify-dataset', 'verify',
    '--dataset-root', str(drive_dataset),
    '--mode', 'metadata',
    '--expected-version', EXPECTED_VERSION,
    '--expected-fingerprint', EXPECTED_FINGERPRINT,
    '--report', str(preflight_dir / 'drive_release_verification.json'),
], check=True)


## 5. Drive → `/content` local SSD staging + full SHA/decode verification

Drive는 영구 저장소로만 사용하고 실제 random-access training은 `/content`에서 수행합니다. 복사는 임시 디렉터리에 이루어지고 full verification이 성공한 뒤에만 최종 경로로 원자적으로 이동합니다.


In [ ]:
subprocess.run([
    'da-daka-verify-dataset', 'stage',
    '--source-root', str(drive_dataset),
    '--destination-root', LOCAL_DATASET_ROOT,
    '--reuse-verified',
    '--expected-version', EXPECTED_VERSION,
    '--expected-fingerprint', EXPECTED_FINGERPRINT,
    '--report', str(preflight_dir / 'local_staging_verification.json'),
], check=True)


## 6. 실제 Panel/Dirt loader smoke test와 dataset 분석


In [ ]:
panel_config = repo / 'training/configs/panel_detector.yaml'
dirt_config = repo / 'training/configs/dirt_segmenter.yaml'
subprocess.run([
    'da-daka-loader-smoke',
    '--dataset-root', LOCAL_DATASET_ROOT,
    '--panel-config', str(panel_config),
    '--dirt-config', str(dirt_config),
    '--expected-version', EXPECTED_VERSION,
    '--expected-fingerprint', EXPECTED_FINGERPRINT,
    '--report', str(preflight_dir / 'loader_smoke.json'),
], check=True)
subprocess.run([
    'da-daka-analyze-dataset',
    '--dataset-root', LOCAL_DATASET_ROOT,
    '--candidates', str(repo / 'training/configs/resolution_candidates.yaml'),
    '--expected-version', EXPECTED_VERSION,
    '--expected-fingerprint', EXPECTED_FINGERPRINT,
    '--output', str(preflight_dir / 'dataset_geometry_analysis.json'),
], check=True)


## 7. Panel Detector training

매 epoch `last.pt`, 개선 시 `best.pt`, history/metadata가 Drive artifact directory에 원자적으로 mirror됩니다. 재개할 때 `PANEL_RESUME`에 Drive의 `last.pt`를 지정하고 같은 `RUN_LABEL`을 사용하세요.


In [ ]:
local_runs = Path(LOCAL_RUNS_ROOT)
local_runs.mkdir(parents=True, exist_ok=True)
if 'panel' in TASKS:
    command = [
        'da-daka-train-panel', '--config', str(panel_config),
        '--dataset-root', LOCAL_DATASET_ROOT,
        '--output-dir', str(local_runs / RUN_LABEL / 'panel'),
        '--artifact-dir', str(drive_results / RUN_LABEL / 'panel'),
        '--expected-dataset-version', EXPECTED_VERSION,
        '--expected-dataset-fingerprint', EXPECTED_FINGERPRINT,
    ]
    if PANEL_RESUME.strip():
        command += ['--resume', PANEL_RESUME]
    subprocess.run(command, check=True, cwd=repo)


## 8. Dirt Segmenter training

validation probability 파일은 `/content`에서 생성한 뒤 하나의 ZIP으로 묶어 Drive에 저장합니다. 이는 Drive에 수천 개 작은 파일을 쓰는 병목을 줄입니다.


In [ ]:
if 'dirt' in TASKS:
    command = [
        'da-daka-train-dirt', '--config', str(dirt_config),
        '--dataset-root', LOCAL_DATASET_ROOT,
        '--output-dir', str(local_runs / RUN_LABEL / 'dirt'),
        '--artifact-dir', str(drive_results / RUN_LABEL / 'dirt'),
        '--expected-dataset-version', EXPECTED_VERSION,
        '--expected-dataset-fingerprint', EXPECTED_FINGERPRINT,
    ]
    if DIRT_RESUME.strip():
        command += ['--resume', DIRT_RESUME]
    subprocess.run(command, check=True, cwd=repo)


## 9. Dirt validation threshold sweep

0.50은 placeholder입니다. 이 report의 false-clean을 포함한 지표를 검토한 뒤 threshold를 승인해야 하며, 이 notebook은 자동으로 threshold를 승인하지 않습니다.


In [ ]:
if 'dirt' in TASKS:
    subprocess.run([
        'da-daka-threshold-sweep',
        '--dataset-root', LOCAL_DATASET_ROOT,
        '--predictions-dir', str(local_runs / RUN_LABEL / 'dirt/validation_probabilities'),
        '--start', '0.05', '--stop', '0.95', '--step', '0.05',
        '--minimum-component-area', '8',
        '--minimum-component-area-ratio', '0.0001',
        '--expected-version', EXPECTED_VERSION,
        '--expected-fingerprint', EXPECTED_FINGERPRINT,
        '--output', str(drive_results / RUN_LABEL / 'dirt/threshold_sweep.json'),
    ], check=True, cwd=repo)


## 10. Locked test evaluation (명시적 승인 후만)

validation으로 해상도/threshold를 선택한 후 아래 플래그와 승인값을 설정해 test split을 1회 평가하세요.


In [ ]:
RUN_LOCKED_TEST_EVALUATION = False
APPROVED_DIRT_THRESHOLD = None
APPROVED_PANEL_SCORE_THRESHOLD = None
if RUN_LOCKED_TEST_EVALUATION:
    if APPROVED_DIRT_THRESHOLD is None or APPROVED_PANEL_SCORE_THRESHOLD is None:
        raise RuntimeError('Locked test evaluation requires both approved thresholds.')
    subprocess.run([
        'da-daka-evaluate-model',
        '--checkpoint', str(drive_results / RUN_LABEL / 'panel/checkpoints/best.pt'),
        '--dataset-root', LOCAL_DATASET_ROOT, '--split', 'test',
        '--score-threshold', str(APPROVED_PANEL_SCORE_THRESHOLD),
        '--output-dir', str(drive_results / RUN_LABEL / 'panel/test-evaluation'),
    ], check=True, cwd=repo)
    subprocess.run([
        'da-daka-evaluate-model',
        '--checkpoint', str(drive_results / RUN_LABEL / 'dirt/checkpoints/best.pt'),
        '--dataset-root', LOCAL_DATASET_ROOT, '--split', 'test',
        '--threshold', str(APPROVED_DIRT_THRESHOLD),
        '--output-dir', str(drive_results / RUN_LABEL / 'dirt/test-evaluation'),
    ], check=True, cwd=repo)
